<div align="center">

# <span style="color:#2E86C1;"> Lab 2: Advanced Data Preparation for Machine Learning</span>

**Author:** [<span style="color:#8E44AD;">Dr. D Bhanu Prakash</span>](https://dbhanuprakash233.github.io)

<img src="https://github.com/dbhanuprakash233/SSSIHL_DBP/blob/main/assets/SssihlLogo.png?raw=true" alt="University Logo" width="80"/>

**<span style="color:#16A085;">Sri Sathya Sai Institute of Higher Learning</span>**  
<span style="color:#5D6D7E;">Prasanthi Nilayam - 515 134, Andhra Pradesh, India.</span>

**Course:** <span style="color:#D35400;">Data Analysis and Visualisation</span>  
**Course Code:** <span style="color:#1ABC9C;">Minor</span>

</div>

This notebook demonstrates an advanced data processing pipeline on the Telco Customer Churn dataset. It applies an **Isolation Forest** to identify multivariate outliers, followed by a **Yeo-Johnson Power Transform** to normalize heavily skewed distributions.

### Step 0: Environment Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import PowerTransformer

# Set plot style for better visuals
sns.set_theme(style="whitegrid")

# Load Telco Churn Dataset from IBM GitHub
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print(df.head(10))

df.describe()

df.info()

### Step 1: Data Selection
Isolate the relevant features for the modeling task and parse out numerical features that require advanced cleaning.

In [ ]:
# 'TotalCharges' is imported as a string due to empty spaces. 
# We coerce it to numeric, which turns empty spaces into NaN (missing values).
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Select key numerical features to demonstrate the pipeline
features = ['tenure', 'MonthlyCharges', 'TotalCharges']
X = df[features].copy()

print(f"Initial Dataset Shape: {X.shape}")

### Step 3: Data Cleaning (Isolation Forest)
Handle missing values first. Then, use **Isolation Forest** to detect and remove multivariate outliers.

In [ ]:
# 1. Impute missing values with the median
X['TotalCharges'] = X['TotalCharges'].fillna(X['TotalCharges'].median())

# 2. Advanced Outlier Detection using Isolation Forest
# contamination=0.02 assumes 2% of our dataset consists of outliers
iso_forest = IsolationForest(contamination=0.02, random_state=42)
outlier_labels = iso_forest.fit_predict(X)

# Add labels to dataframe (1 = normal inlier, -1 = anomaly/outlier)
X['Outlier'] = outlier_labels

# Plotting the Importance of Outlier Detection
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=X, 
    x='tenure', 
    y='TotalCharges', 
    hue='Outlier', 
    palette={1: '#3498db', -1: '#e74c3c'}, 
    alpha=0.6
)
plt.title("Isolation Forest: Detecting Multivariate Outliers")
plt.xlabel("Tenure (Months)")
plt.ylabel("Total Charges ($)")
plt.legend(['Inliers', 'Outliers'])
plt.show()

# Filter out the outliers to produce a clean dataset for transformation
X_clean = X[X['Outlier'] == 1].drop(columns=['Outlier'])
print(f"Cleaned Dataset Shape: {X_clean.shape}")

### Step 4: Data Transformation (Power Transform)
Apply the **Yeo-Johnson Power Transform** to automatically discover the optimal lambda parameter, shifting highly skewed data toward a Gaussian (normal) bell curve.

In [ ]:
# Initialize Yeo-Johnson Power Transformer
pt = PowerTransformer(method='yeo-johnson')

# Fit and transform the cleaned data
X_transformed = pd.DataFrame(pt.fit_transform(X_clean), columns=X_clean.columns)

# Plotting the Importance of Transformation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before Transformation
sns.histplot(X_clean['TotalCharges'], kde=True, ax=axes[0], color='coral')
axes[0].set_title("Before Transformation: Heavily Skewed")
axes[0].set_xlabel("Total Charges (Original Scale)")

# After Transformation
sns.histplot(X_transformed['TotalCharges'], kde=True, ax=axes[1], color='seagreen')
axes[1].set_title("After Yeo-Johnson: Gaussian-like Distribution")
axes[1].set_xlabel("Total Charges (Power Transformed)")

plt.tight_layout()
plt.show()